# Question 3: Stable Diffusion Inpainting (Demo)

This notebook applies a pretrained Diffusion inpainting model to masked dog images. It is used as the advanced generative model in the project. The model is not trained from scratch instead it performs inference using a large pretrained diffusion prior. For this problem, we used a Stable Diffusion model with text-conditioned guidance during inference, so it is a guided diffusion-style approach in practice.

We implemented an advanced generative model for the dog image completion project. Earlier notebooks trained U-Net-based edge and image completion models from scratch. Those two models showed stable pixel recontruction losses but blurry qualitative results, which motivates using a stronger pretrained generative prior.

Here, we use **Stable Diffusion Inpainting** as a demo/advanced model. The model is **not trained from scratch** instead we use a pretrained diffusion inpainting pipeline to complete masked regions in dog images.

**Goal:** compare the reconstruction-loss CNN-based approach with a pretrained diffusion-based inpainting model that can generate sharper and more realistic dog texture.

## 1. Runtime and Installation


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
!pip install diffusers transformers accelerate safetensors -q

In [ ]:
from pathlib import Path
import os
import random
import math
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image, ImageDraw, ImageFilter

import torch
from tqdm.auto import tqdm
from diffusers import StableDiffusionInpaintPipeline

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
DTYPE = torch.float16 if DEVICE == "cuda" else torch.float32

print("Using device:", DEVICE)
print("Torch dtype:", DTYPE)

## 2. Project Paths

Update `BASE_DIR` if your project folder is stored somewhere else. The code assumes the Stanford Dogs image folder is inside your project directory.

The notebook will save Stable Diffusion outputs into `results/stable_diffusion_inpainting`.

In [ ]:
BASE_DIR = Path("/content/drive/MyDrive/Adv CV/CV Project Folder")
IMAGE_DIR = BASE_DIR / "Images"
RESULT_DIR = BASE_DIR / "results" / "stable_diffusion_inpainting"
RESULT_DIR.mkdir(parents=True, exist_ok=True)

print("BASE_DIR exists:", BASE_DIR.exists())
print("IMAGE_DIR exists:", IMAGE_DIR.exists())
print("Saving results to:", RESULT_DIR)

In [ ]:
image_paths = sorted(list(IMAGE_DIR.rglob("*.jpg")))
print("Number of dog images found:", len(image_paths))

if len(image_paths) == 0:
    raise FileNotFoundError("No .jpg images found. Please check IMAGE_DIR.")

print("Example image path:", image_paths[0])

## 3. Helper Functions

Stable Diffusion inpainting expects:

- an RGB image
- a mask image
- **white pixels = region to inpaint**
- **black pixels = region to keep unchanged**

The helper functions below create synthetic occlusion masks and visualization grids. These masks simulate objects blocking parts of the dog image.

In [ ]:
def load_image(path, size=512):
    """Load an image as RGB and resize to the Stable Diffusion standard input size."""
    return Image.open(path).convert("RGB").resize((size, size))


def create_rect_mask(size=512, min_ratio=0.20, max_ratio=0.42):
    """
    Create a rectangular white mask for inpainting.
    White = missing region to fill; black = preserve original image.
    """
    mask = Image.new("L", (size, size), 0)
    draw = ImageDraw.Draw(mask)

    w = random.randint(int(size * min_ratio), int(size * max_ratio))
    h = random.randint(int(size * min_ratio), int(size * max_ratio))
    x = random.randint(0, size - w)
    y = random.randint(0, size - h)

    draw.rectangle([x, y, x + w, y + h], fill=255)
    return mask


def create_center_face_mask(size=512, box_ratio=0.36):
    """
    Create a centered mask. This is useful for dog-face examples where we want
    to test whether the model can complete eyes, nose, or fur structure.
    """
    mask = Image.new("L", (size, size), 0)
    draw = ImageDraw.Draw(mask)

    box = int(size * box_ratio)
    x1 = (size - box) // 2
    y1 = int(size * 0.27)
    x2 = x1 + box
    y2 = y1 + box

    draw.rectangle([x1, y1, x2, y2], fill=255)
    return mask


def create_irregular_mask(size=512, max_strokes=5):
    """
    Create an irregular brush-stroke mask. This is more realistic than a pure rectangle,
    but it may be harder to evaluate consistently.
    """
    mask = Image.new("L", (size, size), 0)
    draw = ImageDraw.Draw(mask)

    for _ in range(random.randint(2, max_strokes)):
        points = []
        x, y = random.randint(0, size - 1), random.randint(0, size - 1)
        for _ in range(random.randint(3, 7)):
            x = min(max(x + random.randint(-80, 80), 0), size - 1)
            y = min(max(y + random.randint(-80, 80), 0), size - 1)
            points.append((x, y))
        width = random.randint(18, 45)
        draw.line(points, fill=255, width=width)
        for px, py in points:
            r = random.randint(10, 30)
            draw.ellipse([px-r, py-r, px+r, py+r], fill=255)

    return mask.filter(ImageFilter.GaussianBlur(radius=1))


def apply_mask_for_display(image, mask):
    """Create a masked input visualization by painting the missing region black."""
    image_np = np.array(image).copy()
    mask_np = np.array(mask)
    image_np[mask_np > 127] = 0
    return Image.fromarray(image_np)


def show_images(images, titles, figsize=(18, 5), save_path=None):
    """Display a horizontal comparison grid and optionally save it."""
    n = len(images)
    fig, axes = plt.subplots(1, n, figsize=figsize)
    if n == 1:
        axes = [axes]
    for ax, img, title in zip(axes, images, titles):
        ax.imshow(img, cmap="gray" if img.mode == "L" else None)
        ax.set_title(title)
        ax.axis("off")
    plt.tight_layout()
    if save_path is not None:
        plt.savefig(save_path, dpi=200, bbox_inches="tight")
        print("Saved figure to:", save_path)
    plt.show()

## 4. Preview One Masked Example

Before running diffusion, we verify that the image and mask are correctly formatted. The mask should be white where we want Stable Diffusion to generate new pixels.

In [ ]:
random.seed(42)
example_path = image_paths[0]
image = load_image(example_path, size=512)
mask = create_center_face_mask(size=512, box_ratio=0.34)
masked_display = apply_mask_for_display(image, mask)

show_images(
    [image, mask, masked_display],
    ["Original Image", "Mask: White = Inpaint", "Masked Input Display"],
    figsize=(12, 4)
)

## 5. Load Stable Diffusion Inpainting Pipeline

This is the advanced model stage. We use a pretrained Stable Diffusion inpainting model rather than training a diffusion model from scratch.

This is appropriate for the project because the goal is to demonstrate how a pretrained generative model improves image completion quality compared with a small U-Net trained only with pixel reconstruction loss.


In [ ]:
MODEL_ID = "runwayml/stable-diffusion-inpainting"

pipe = StableDiffusionInpaintPipeline.from_pretrained(
    MODEL_ID,
    torch_dtype=DTYPE,
    safety_checker=None,
).to(DEVICE)

# Memory optimizations for Colab GPUs
pipe.enable_attention_slicing()
try:
    pipe.enable_xformers_memory_efficient_attention()
    print("xFormers memory efficient attention enabled.")
except Exception as e:
    print("xFormers not enabled; continuing without it.")

print("Loaded model:", MODEL_ID)

## 6. Single-Image Stable Diffusion Inpainting Demo

Now, let's run a pretrained diffusion model on one masked dog image.

Prompting matters because diffusion models use text guidance. The prompt encourages the model to generate realistic dog fur and facial texture in the missing region.

In [ ]:
prompt = "a realistic photo of a fluffy dog, natural fur texture, detailed dog face, high quality"
negative_prompt = "blurry, distorted, cartoon, extra eyes, deformed face, low quality, artifacts"

generator = torch.Generator(device=DEVICE).manual_seed(123)

result = pipe(
    prompt=prompt,
    negative_prompt=negative_prompt,
    image=image,
    mask_image=mask,
    guidance_scale=7.5,
    num_inference_steps=35,
    strength=0.95,
    generator=generator,
).images[0]

save_path = RESULT_DIR / "stable_diffusion_single_example.png"
show_images(
    [image, masked_display, mask, result],
    ["Ground Truth", "Masked Input", "Mask", "Stable Diffusion Completion"],
    figsize=(18, 5),
    save_path=save_path
)

result.save(RESULT_DIR / "stable_diffusion_single_output.png")

## 7. Batch Demo on Multiple Images

This is still a **demo/inference workflow**, not full diffusion training. It runs the pretrained Stable Diffusion inpainting model on a small set of dog images and saves comparison figures.


In [ ]:
def run_sd_inpainting_demo(
    image_paths,
    n_examples=6,
    size=512,
    mask_type="center",  # options: "center", "rect", "irregular"
    seed=123,
    num_inference_steps=35,
    guidance_scale=7.5,
):
    random.seed(seed)
    selected_paths = random.sample(image_paths, min(n_examples, len(image_paths)))
    records = []

    for i, path in enumerate(tqdm(selected_paths, desc="Stable Diffusion inpainting")):
        img = load_image(path, size=size)

        if mask_type == "center":
            m = create_center_face_mask(size=size, box_ratio=0.34)
        elif mask_type == "rect":
            m = create_rect_mask(size=size)
        elif mask_type == "irregular":
            m = create_irregular_mask(size=size)
        else:
            raise ValueError("mask_type must be 'center', 'rect', or 'irregular'")

        masked = apply_mask_for_display(img, m)
        gen = torch.Generator(device=DEVICE).manual_seed(seed + i)

        out = pipe(
            prompt=prompt,
            negative_prompt=negative_prompt,
            image=img,
            mask_image=m,
            guidance_scale=guidance_scale,
            num_inference_steps=num_inference_steps,
            strength=0.95,
            generator=gen,
        ).images[0]

        out_path = RESULT_DIR / f"sd_output_{i}.png"
        fig_path = RESULT_DIR / f"sd_comparison_{i}.png"
        out.save(out_path)

        show_images(
            [img, masked, m, out],
            ["Ground Truth", "Masked Input", "Mask", "Stable Diffusion Completion"],
            figsize=(18, 5),
            save_path=fig_path
        )

        records.append({
            "index": i,
            "image_path": str(path),
            "output_path": str(out_path),
            "figure_path": str(fig_path),
            "mask_type": mask_type,
            "prompt": prompt,
            "negative_prompt": negative_prompt,
            "num_inference_steps": num_inference_steps,
            "guidance_scale": guidance_scale,
        })

    return pd.DataFrame(records)

In [ ]:
results_df = run_sd_inpainting_demo(
    image_paths=image_paths,
    n_examples=4,
    size=512,
    mask_type="center",
    seed=123,
    num_inference_steps=35,
    guidance_scale=7.5,
)

results_csv = RESULT_DIR / "stable_diffusion_results.csv"
results_df.to_csv(results_csv, index=False)
print("Saved results table to:", results_csv)
results_df.head()

## 8. Quantitative Evaluation

Diffusion inpainting is designed to generate **plausible** completions, not necessarily pixel-perfect copies of the original image. Therefore, pixel metrics such as L1, MSE, and PSNR can be useful but should not be the only evaluation.

For evaluation we should use both:

1. **Qualitative comparison:** masked input, U-Net result, Stable Diffusion result, ground truth.
2. **Quantitative comparison:** L1/MSE/PSNR on the masked region if using synthetic masks.

In [ ]:
def compute_mask_region_metrics(original, completed, mask):
    """
    Compute simple pixel metrics only inside the masked region.
    Lower L1/MSE is better; higher PSNR is better.
    """
    orig = np.asarray(original).astype(np.float32) / 255.0
    comp = np.asarray(completed).astype(np.float32) / 255.0
    m = np.asarray(mask).astype(np.float32) / 255.0
    m = (m > 0.5).astype(np.float32)[..., None]

    diff = (orig - comp) * m
    denom = np.maximum(m.sum() * 3, 1)

    l1 = np.abs(diff).sum() / denom
    mse = (diff ** 2).sum() / denom
    psnr = 20 * math.log10(1.0 / math.sqrt(mse + 1e-8))

    return {"masked_l1": float(l1), "masked_mse": float(mse), "masked_psnr": float(psnr)}

# Example metric for the single demo image
metrics = compute_mask_region_metrics(image, result, mask)
metrics